Outputs `genes.csv` as final data.

Outputs `cells.parquet` and `count.parquet` as intermediate data.

In [1]:
import pandas as pd

In [7]:
# Paths
dataset = "MERSCOPE_9m_WT_R2"
data_path = f"../data/{dataset}/"

In [9]:
# Read count matrix
counts = pd.read_csv(data_path + "raw_data/count.csv")
counts.head()

,cell,Igf2,Pdgfra,S100a6,Col6a1,Col1a1,Grik5,Calb2,Eno2,Stx1a,...,Blank-16,Blank-17,Blank-18,Blank-19,Blank-20,Blank-21,Blank-22,Blank-23,Blank-24,Blank-25
0,1615410000031100011,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,1615410000032100002,3,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1615410000033100003,10,5,0,1,1,1,0,2,0,...,0,0,0,0,0,0,0,0,0,0
3,1615410000033100004,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,1615410000033100005,2,0,0,1,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [10]:
# Remove blank & neg-control genes
genes = list(counts.columns)
genes.pop(0)
genes = [i for i in genes if (i.startswith("Blank") == False) and (i.startswith("NegControl") == False)]
pd.DataFrame({"genes": genes}).to_csv(data_path + "processed_data/genes.csv", index=0)

In [11]:
# Read metadata and filter cells
cells = pd.read_csv(data_path + "raw_data/metadata.csv")
cells = cells[cells["transcript_count"] > 25]
cell_id = list(cells["EntityID"])
cells = cells.iloc[:, 1:]
cells.insert(0, "cell_id", cell_id)
cells.to_parquet(data_path + "intermediate_data/cells.parquet")
cells.head()

,cell_id,fov,volume,center_x,center_y,min_x,min_y,max_x,max_y,anisotropy,...,perimeter_area_ratio,solidity,PolyT_raw,PolyT_high_pass,Snap25_raw,Snap25_high_pass,DAPI_raw,DAPI_high_pass,Apoe_raw,Apoe_high_pass
1,1615410000032100002,467,491.246403,4266.282753,155.603452,4261.347019,151.524553,4273.093535,160.164386,1.381873,...,0.640009,4.278967,250429112.0,1.932401e+06,22066040.0,3.345996e+05,157141947.0,1.736642e+06,84337629.0,2.806189e+06
2,1615410000033100003,467,1599.944395,4387.216621,128.710407,4378.506809,121.611081,4395.995979,139.790899,1.268078,...,0.338940,4.646751,964980294.0,6.715941e+06,71964111.0,1.069581e+06,662377195.0,4.809386e+06,237214201.0,6.052794e+06
3,1615410000033100004,467,137.357358,4388.652988,134.806582,4383.650118,131.991256,4393.377112,137.645893,1.914073,...,0.800724,2.458747,83546996.0,9.260226e+05,6897222.0,1.542989e+05,47857620.0,7.412331e+05,29365442.0,9.091303e+05
4,1615410000033100005,467,359.214532,4371.377950,142.992474,4366.741567,138.566753,4375.660243,148.866273,1.179635,...,0.591299,3.049720,141586528.0,1.453302e+06,15410776.0,2.650549e+05,48529565.0,8.387673e+05,65476586.0,2.530470e+06
5,1615410000033100006,467,893.902197,4381.392631,142.470371,4374.849180,137.405698,4388.609666,150.113971,1.272624,...,0.416986,4.656450,399126454.0,3.364266e+06,38367043.0,6.057693e+05,197714138.0,2.317680e+06,158245578.0,5.026090e+06


In [12]:
# Update count data and add cell id
counts = counts[counts["cell"].isin(cell_id)]
counts = counts[counts.columns.intersection(genes)]
counts.insert(0, "cell_id", cell_id)
counts.to_parquet(data_path + "intermediate_data/count.parquet")
counts.head()

,cell_id,Igf2,Pdgfra,S100a6,Col6a1,Col1a1,Grik5,Calb2,Eno2,Stx1a,...,Olfr287,Prdm8,Cryab,Rmst,Psd,Rspo2,Sox10,Fhod3,Cacna1a,Aqp4
1,1615410000032100002,3,0,0,0,0,0,0,0,0,...,0,0,0,0,3,0,0,0,0,5
2,1615410000033100003,10,5,0,1,1,1,0,2,0,...,0,0,0,0,2,0,0,0,0,24
3,1615410000033100004,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,3
4,1615410000033100005,2,0,0,1,0,1,0,0,0,...,0,0,0,1,1,0,0,0,0,2
5,1615410000033100006,0,0,1,0,0,2,0,0,0,...,1,1,0,0,7,0,0,0,0,14
